In [2]:
import spacy 
model_dir = spacy.util.get_package_path('xx_ent_wiki_sm')
print(model_dir)


/Users/ricardofernandezgasca/Library/CloudStorage/OneDrive-Personnel/Estudios/ESTUDIOS2024/NOM005Bitron24/proyecto/myenv/lib/python3.10/site-packages/xx_ent_wiki_sm


/Users/ricardofernandezgasca/Library/CloudStorage/OneDrive-Personnel/Estudios/ESTUDIOS2024/NOM005Bitron24/proyecto/myenv/lib/python3.10/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'xx_ent_wiki_sm' (3.7.0) was trained with spaCy v3.7.0 and may not be 100% compatible with the current version (3.8.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [ ]:
from preprocess import DocumentPreprocessor
from extract_info import extract_info_from_hds_txt
from excel_postprocess import GeneradorTablaSustQ
import openai

file_path = 'ejemplo/123- Freon 407A (R-407A) Refrigerante.pdf'
config_path = 'config.json'
client=openai.OpenAI()
doc=DocumentPreprocessor().extract_text(file_path)



PDF is valid: ejemplo/123- Freon 407A (R-407A) Refrigerante.pdf (pages: 16)
Native PDF extraction failed, trying OCR...


TypeError: extract_info_from_hds_txt() missing 1 required positional argument: 'client'

In [6]:
info=extract_info_from_hds_txt(doc,client)
list_hds=[]
hds_dict=info.model_dump()
list_hds.append(hds_dict)

flat_table=GeneradorTablaSustQ().flatten_hds_data(list_hds)

Config path initialized as: data_sets


In [7]:
print(flat_table)

[{'Archivo': None, 'Nombre de la Sustancia Química': 'Freon 407C (R-407C) Refrigerante', 'Idioma de la HDS': 'Español', 'Estado Físico': 'Gaseoso', 'Sujeta a RETC': 'SI', 'Sujeta a GEI': 'REVISAR', 'pH de la sustancia': None, 'Palabra de Advertencia': 'Atención', 'Indicaciones de toxicología': 'Puede causar arritmia cardíaca. El contacto con el líquido o gas refrigerado puede causar quemaduras frías y congelamiento.', 'Pictogramas': 'bomba_explotando, llama, llama_sobre_circulo, cilindro_de_gas, corrosion, calavera_tibias_cruzadas, signo_de_exclamacion, peligro_para_la_salud, medio_ambiente', 'Olor': 'ligero, similar al éter', 'Color': 'incoloro', 'Propiedades Explosivas': 'No explosivo', 'Propiedades Comburentes': 'La sustancia o mezcla no se clasifica como oxidante.', 'Tamaño de Partícula': None, 'Pictogramas Explosivos': None, 'Pictogramas Inflamables': None, 'Pictogramas Comburentes': None, 'Pictogramas Gases Comprimidos': 'X', 'Pictogramas Corrosivos': None, 'Pictogramas Toxicidad

In [13]:
import pandas as pd
# Convert flat_table to a DataFrame
df_flat = pd.DataFrame(flat_table)

# Filter rows where "Componente GEI" is not None
df_filtered = df_flat[df_flat['Componente GEI'].notnull()]

# Select desired columns
df_result = df_filtered[['Nombre de la Sustancia Química', 'Nombre del Componente', 'Porcentaje del Componente', 'Potencial de Calentamiento Global']]

# Convert 'Porcentaje del Componente' to numeric
df_result.loc[:, 'Porcentaje del Componente'] = pd.to_numeric(df_result['Porcentaje del Componente'], errors='coerce')

# Calculate 'Factor GEI'
df_result.loc[:, 'Factor GEI'] = df_result['Porcentaje del Componente'] * df_result['Potencial de Calentamiento Global']/100

# Display the DataFrame
df_result

/var/folders/z8/5xwgt9dn43n0_s6896y6y4yc0000gn/T/ipykernel_99764/1456707511.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result.loc[:, 'Factor GEI'] = df_result['Porcentaje del Componente'] * df_result['Potencial de Calentamiento Global']/100


,Nombre de la Sustancia Química,Nombre del Componente,Porcentaje del Componente,Potencial de Calentamiento Global,Factor GEI
0,Freon 407C (R-407C) Refrigerante,"1,1,1,2-Tetrafluoroetano",52,1300,676.0
1,Freon 407C (R-407C) Refrigerante,Pentafluoroetano,25,3170,792.5
2,Freon 407C (R-407C) Refrigerante,Difluorometano,23,677,155.71


In [15]:
import pandas as pd

# Assuming df_result already has the 'Factor GEI' calculated
# Here's the code from your previous steps:

# Convert flat_table to a DataFrame
df_flat = pd.DataFrame(flat_table)

# Filter rows where "Componente GEI" is not None
df_filtered = df_flat[df_flat['Componente GEI'].notnull()]

# Select desired columns
df_result = df_filtered[['Nombre de la Sustancia Química',
                         'Nombre del Componente',
                         'Porcentaje del Componente',
                         'Potencial de Calentamiento Global']]

# Convert 'Porcentaje del Componente' to numeric
df_result.loc[:, 'Porcentaje del Componente'] = pd.to_numeric(df_result['Porcentaje del Componente'], errors='coerce')

# Calculate 'Factor GEI'
df_result.loc[:, 'Factor GEI'] = df_result['Porcentaje del Componente'] * df_result['Potencial de Calentamiento Global'] / 100
print(df_result)
# Now, group by 'Nombre de la Sustancia Química' and sum 'Factor GEI'
df_total_pcg = df_result.groupby('Nombre de la Sustancia Química')['Factor GEI'].sum().reset_index()

# Rename 'Factor GEI' column to 'PCG' to represent the total PCG per substance
df_total_pcg.rename(columns={'Factor GEI': 'PCG'}, inplace=True)

# Display the result
print(df_total_pcg)

     Nombre de la Sustancia Química  ... Factor GEI
0  Freon 407C (R-407C) Refrigerante  ...      676.0
1  Freon 407C (R-407C) Refrigerante  ...      792.5
2  Freon 407C (R-407C) Refrigerante  ...     155.71

[3 rows x 5 columns]
     Nombre de la Sustancia Química      PCG
0  Freon 407C (R-407C) Refrigerante  1624.21


/var/folders/z8/5xwgt9dn43n0_s6896y6y4yc0000gn/T/ipykernel_99764/3558545130.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_result.loc[:, 'Factor GEI'] = df_result['Porcentaje del Componente'] * df_result['Potencial de Calentamiento Global'] / 100
